In [1]:
import sys
sys.path.insert(0, '/Users/polina/spatial-disaggregation')

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

df_full = pd.read_parquet('/Users/polina/spatial-disaggregation/data/raw/Массив Данных в формате Parquet/Показатели по регионам.parquet')

from src.masks.pipeline import run_pipeline, AVAILABLE_MASKS
print(f"Доступные маски: {list(AVAILABLE_MASKS.keys())}")
print(f"Всего: {len(AVAILABLE_MASKS)}")

Доступные маски: ['baseline', 'worldpop', 'regression', 'distance_to_city', 'distance_to_center']
Всего: 5


In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd

df_full = pd.read_parquet('/Users/polina/spatial-disaggregation/data/raw/Массив Данных в формате Parquet/Показатели по регионам.parquet')

FINAL_WEIGHTS = {
    'regression': 0.5,
    'worldpop': 0.3,
    'distance_to_city': 0.1,
    'distance_to_center': 0.1,
}

INDICATORS = {
    'Y477110039': ('НИОКР', 'niokr'),
    'Y477090007': ('Отгрузка', 'otgr'),
}

REGIONS_FILES = {
    'Московская область': ('grid_moscow_1km_features.gpkg', 'moscow'),
    'Краснодарский край': ('grid_krasnodar_1km_features.gpkg', 'krasnodar'),
    'Республика Саха (Якутия)': ('grid_yakutia_center_1km_features.gpkg', 'yakutia'),
}

summary_rows = []

for ind_code, (ind_name, ind_short) in INDICATORS.items():
    for region, (grid_file, short_name) in REGIONS_FILES.items():
        grid = gpd.read_file(f'/Users/polina/spatial-disaggregation/data/processed/{grid_file}')
        
        val = df_full[
            (df_full['indicator_code'] == ind_code) & 
            (df_full['object_level'] == 'Регион') & 
            (df_full['year'] == 2023) &
            (df_full['object_name'] == region)
        ]['indicator_value'].values
        
        if len(val) == 0:
            print(f"НЕТ ДАННЫХ: {region} {ind_name}")
            continue
        
        regional_value = float(val[0])
        
        res = run_pipeline(
            grid=grid,
            regional_value=regional_value,
            indicator_code=ind_code,
            masks_to_use=['regression', 'worldpop', 'distance_to_city', 'distance_to_center'],
            mask_weights=FINAL_WEIGHTS
        )
        
        v = res['values']
        top10 = int(len(v) * 0.1)
        top10_share = np.sort(v)[-top10:].sum() / v.sum() * 100
        
        sorted_v = np.sort(v)
        n = len(sorted_v)
        gini = (2 * np.sum((np.arange(1, n+1)) * sorted_v) - (n+1) * sorted_v.sum()) / (n * sorted_v.sum())
        
        summary_rows.append({
            'Показатель': ind_short,
            'Регион': short_name,
            'Сумма': round(regional_value, 1),
            'Джини': round(gini, 3),
            'Top10%': round(top10_share, 1),
            'Сохранение': res['sum_preserved']
        })
        
        grid['value'] = v
        out_file = f'/Users/polina/spatial-disaggregation/results/{ind_short}_{short_name}_5masks.gpkg'
        grid[['cell_id', 'value', 'geometry']].to_file(out_file, driver='GPKG')

summary_df = pd.DataFrame(summary_rows)
print("=== Финальная таблица (5 масок, 4 в композиции) ===\n")
print(summary_df.to_string(index=False))

summary_df.to_csv('/Users/polina/spatial-disaggregation/results/summary_5masks.csv', index=False)

=== Финальная таблица (5 масок, 4 в композиции) ===

Показатель    Регион    Сумма  Джини  Top10%  Сохранение
     niokr    moscow 202367.1  0.532    35.1        True
     niokr krasnodar   9058.4  0.809    69.1        True
     niokr   yakutia   4387.4  0.849    76.9        True
      otgr    moscow    195.0  0.519    33.2        True
      otgr krasnodar     50.8  0.788    65.5        True
      otgr   yakutia     10.8  0.841    75.6        True


In [3]:
print("=== Ablation Study на 5 масках (Московская область, НИОКР) ===\n")

NIOKR_MO = 202367.1
grid_mo = gpd.read_file('/Users/polina/spatial-disaggregation/data/processed/grid_moscow_1km_features.gpkg')

configurations = [
    ('Все 5 масок (полная композиция)', ['regression', 'worldpop', 'distance_to_city', 'distance_to_center'], 
     {'regression': 0.5, 'worldpop': 0.3, 'distance_to_city': 0.1, 'distance_to_center': 0.1}),
    ('Без регрессии', ['worldpop', 'distance_to_city', 'distance_to_center'], 
     {'worldpop': 0.5, 'distance_to_city': 0.25, 'distance_to_center': 0.25}),
    ('Без WorldPop', ['regression', 'distance_to_city', 'distance_to_center'], 
     {'regression': 0.7, 'distance_to_city': 0.15, 'distance_to_center': 0.15}),
    ('Без distance_to_city', ['regression', 'worldpop', 'distance_to_center'], 
     {'regression': 0.6, 'worldpop': 0.3, 'distance_to_center': 0.1}),
    ('Без distance_to_center', ['regression', 'worldpop', 'distance_to_city'], 
     {'regression': 0.6, 'worldpop': 0.3, 'distance_to_city': 0.1}),
    ('Только регрессия', ['regression'], None),
    ('Только WorldPop', ['worldpop'], None),
    ('Только baseline', ['baseline'], None),
]

ablation_rows = []

for config_name, masks_used, weights in configurations:
    res = run_pipeline(
        grid=grid_mo,
        regional_value=NIOKR_MO,
        indicator_code='Y477110039',
        masks_to_use=masks_used,
        mask_weights=weights
    )
    
    v = res['values']
    top10 = int(len(v) * 0.1)
    top10_share = np.sort(v)[-top10:].sum() / v.sum() * 100
    
    sorted_v = np.sort(v)
    n = len(sorted_v)
    gini = (2 * np.sum((np.arange(1, n+1)) * sorted_v) - (n+1) * sorted_v.sum()) / (n * sorted_v.sum())
    
    ablation_rows.append({
        'Конфигурация': config_name,
        'Джини': round(gini, 3),
        'Top10%': round(top10_share, 1),
        'Сохранение': res['sum_preserved']
    })

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))

ablation_df.to_csv('/Users/polina/spatial-disaggregation/results/ablation_5masks_mo.csv', index=False)

=== Ablation Study на 5 масках (Московская область, НИОКР) ===

                   Конфигурация  Джини  Top10%  Сохранение
Все 5 масок (полная композиция)  0.532    35.1        True
                  Без регрессии  0.509    31.7        True
                   Без WorldPop  0.514    32.4        True
           Без distance_to_city  0.813    69.7        True
         Без distance_to_center  0.524    33.8        True
               Только регрессия  0.954    94.7        True
                Только WorldPop  0.910    85.3        True
                Только baseline  0.018    10.2        True
